In [1]:
# [COLAB SETUP]
import sys
import os

if "google.colab" in sys.modules:
    print("Running in Google Colab. Setting up environment...")
    
    # Mount Google Drive to persist the datasets and cloned repository
    from google.colab import drive
    drive.mount('/content/drive')
    
    repo_path = '/content/drive/MyDrive/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit'
    
    if not os.path.exists(repo_path):
        print(f"Cloning repository into {repo_path}...")
        os.makedirs('/content/drive/MyDrive', exist_ok=True)
        os.system(f'git clone https://github.com/Maleesha-K/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit.git {repo_path}')
        
    os.chdir(repo_path + '/data_pipeline')
    print("Installing dependencies...")
    os.system('pip install -q pandas scikit-learn fasttext huggingface_hub')
    print("Setup complete!")


In [2]:
input_dir = 'datasets/preprocessed'
output_dir = 'datasets/benchmark_results'


In [3]:
import os
# Auto-resolve the project root if running manually
if not os.path.exists("Makefile") and os.path.exists("../../Makefile"):
    os.chdir("../../")

import json
import glob
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

TARGET_LANGUAGES = {
    "eng": "eng_Latn",  # English (baseline)
    "sin": "sin_Sinh",  # Sinhala
    "san": "san_Deva",  # Sanskrit
    "tam": "tam_Taml",  # Tamil
    "hin": "hin_Deva",  # Hindi
    "ben": "ben_Beng",  # Bengali
    "arb": "arb_Arab",  # Arabic (Modern Standard)
    "fra": "fra_Latn",  # French
    "deu": "deu_Latn",  # German
}

def load_dataset(file_path):
    print(f"\nLoading {os.path.basename(file_path)}...")
    records = []
    with open(file_path, encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            if row.get("label") in TARGET_LANGUAGES:
                records.append(row)
    df = pd.DataFrame(records)
    if not df.empty:
        df["flores_label"] = df["label"].map(TARGET_LANGUAGES)
        print(f"Loaded {len(df)} rows across {df['label'].nunique()} target languages")
    else:
        print("No matching target languages found in this dataset.")
    return df

def evaluate_and_save(results, model_name, dataset_name, target_labels):
    acc = accuracy_score(results["true_label"], results["predicted_label"])
    macro_f1 = f1_score(
        results["true_label"], results["predicted_label"],
        average="macro", labels=target_labels,
    )

    print("\n" + "=" * 48)
    print(f"ZERO-SHOT BENCHMARK RESULTS ({model_name} on {dataset_name})")
    print("=" * 48)
    print(f"Accuracy:  {acc * 100:.2f}%")
    print(f"Macro F1:  {macro_f1 * 100:.2f}%")
    print("=" * 48)
    print("\nPer-language breakdown:\n")
    print(classification_report(
        results["true_label"], results["predicted_label"],
        labels=target_labels, digits=4,
    ))

    os.makedirs(output_dir, exist_ok=True)
    out_file = os.path.join(output_dir, f"{model_name.replace(' ', '_').replace('-', '_').lower()}_{dataset_name}.csv")
    results.to_csv(out_file, index=False)
    print(f"\nSaved predictions to {out_file}\n")
    return results

dataset_files = glob.glob(os.path.join(input_dir, "*.jsonl"))
if not dataset_files:
    print(f"No datasets found in {input_dir}.")


In [4]:
import fasttext
from huggingface_hub import hf_hub_download

print("Downloading GlotLID v3 model from Hugging Face (~1.6GB)...")
model_path = hf_hub_download(repo_id="cis-lmu/glotlid", filename="model.bin")
print("Loading GlotLID model...")
model = fasttext.load_model(model_path)
model_name = "GlotLID v3"
target_labels = sorted(set(TARGET_LANGUAGES.values()))

for file_path in dataset_files:
    dataset_name = os.path.splitext(os.path.basename(file_path))[0]
    df = load_dataset(file_path)
    if df.empty: continue
    
    texts = df["text"].astype(str).str.replace("\n", " ").tolist()
    print(f"Evaluating {len(texts)} samples with {model_name}...")
    preds, _ = model.predict(texts, k=1)

    results = df[["text", "label", "source"]].copy()
    results["true_label"] = df["flores_label"]
    results["predicted_label"] = [p[0].replace("__label__", "") for p in preds]

    evaluate_and_save(results, model_name, dataset_name, target_labels)


Loading GlotLID model...

Loading flores_plus.jsonl...
Loaded 12116 rows across 9 target languages
Evaluating 12116 samples with GlotLID v3...

ZERO-SHOT BENCHMARK RESULTS (GlotLID v3 on flores_plus)
Accuracy:  79.05%
Macro F1:  82.15%

Per-language breakdown:

              precision    recall  f1-score   support

    arb_Arab     0.9989    0.4343    0.6054      2024
    ben_Beng     1.0000    0.9990    0.9995      1012
    deu_Latn     1.0000    1.0000    1.0000      1012
    eng_Latn     1.0000    1.0000    1.0000      1012
    fra_Latn     1.0000    1.0000    1.0000      1012
    hin_Deva     0.9980    0.9970    0.9975      1012
    san_Deva     0.0000    0.0000    0.0000      1327
    sin_Sinh     0.6651    0.9770    0.7914      2693
    tam_Taml     1.0000    1.0000    1.0000      1012

   micro avg     0.8782    0.7905    0.8321     12116
   macro avg     0.8513    0.8230    0.8215     12116
weighted avg     0.8157    0.7905    0.7779     12116


Saved predictions to datasets/be

/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipel

Loaded 74052 rows across 9 target languages
Evaluating 74052 samples with GlotLID v3...

ZERO-SHOT BENCHMARK RESULTS (GlotLID v3 on commonlid)
Accuracy:  88.46%
Macro F1:  82.98%

Per-language breakdown:

              precision    recall  f1-score   support

    arb_Arab     0.9997    0.9291    0.9631     26152
    ben_Beng     1.0000    0.9592    0.9792      1886
    deu_Latn     0.9877    0.8636    0.9215      7553
    eng_Latn     0.9913    0.8679    0.9255     27461
    fra_Latn     0.9806    0.8763    0.9255      3233
    hin_Deva     0.9940    0.9550    0.9741      3666
    san_Deva     0.0000    0.0000    0.0000      1327
    sin_Sinh     0.6651    0.9770    0.7914      2693
    tam_Taml     0.9759    1.0000    0.9878        81

   micro avg     0.9745    0.8846    0.9274     74052
   macro avg     0.8438    0.8253    0.8298     74052
weighted avg     0.9642    0.8846    0.9208     74052


Saved predictions to datasets/benchmark_results/glotlid_v3_commonlid.csv


Loading wili-2

/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/vihanga/Desktop/programming/Sinhala-Script-Language-Identification-LangID-for-Sinhala-Pali-and-Sanskrit/data_pipeline/.ven